In [29]:
import os
from getpass import getpass
from fastembed import TextEmbedding
from pinecone import Pinecone
from langchain_openai import ChatOpenAI
from pathlib import Path
import urllib.request
import zipfile


In [32]:
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

if "PINECONE_API_KEY" not in os.environ:
    os.environ["PINECONE_API_KEY"] = getpass("PINECONE_API_KEY: ")

In [33]:
# connect pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index("sms-spam-bge384")

# same embed model as Part 1
emb_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

def retrieve_sms(query: str, k: int = 5):
    q_vec = next(emb_model.embed([query])).tolist()
    res = index.query(vector=q_vec, top_k=k, include_metadata=True)
    return res["matches"]

# quick test
print(retrieve_sms("win cash prize", k=5))


[{'id': '4047',
 'metadata': {'label': 'spam'},
 'score': 0.878326416,
 'values': []}, {'id': '576', 'metadata': {'label': 'spam'}, 'score': 0.817485809, 'values': []}, {'id': '650', 'metadata': {'label': 'spam'}, 'score': 0.795520782, 'values': []}, {'id': '930', 'metadata': {'label': 'spam'}, 'score': 0.793316, 'values': []}, {'id': '1970',
 'metadata': {'label': 'spam'},
 'score': 0.774288237,
 'values': []}]


get the data from kaggle

In [34]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

path = data_dir / "SMSSpamCollection"

if not path.exists():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
    zip_path = data_dir / "smsspamcollection.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)

In [35]:
rows = []
with Path("data/SMSSpamCollection").open("r", encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)
        rows.append({"label": label, "text": text})

print(len(rows), rows[0])

5574 {'label': 'ham', 'text': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'}


### Add the 5 questions

In [49]:
questions = [
    "How can I tell if a text message is spam? Give 3 signs.",
    "Write a short reply that refuses a spam offer politely.",
    "If a message asks me to call a number to claim a prize, what should I do?",
    "What are common topics spam texts talk about? Give 3 examples.",
    "Give a simple checklist I can follow before clicking a link in a text message."
]


### Baseline A: Original LLM (no RAG)

In [50]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def no_rag(q: str) -> str:
    return llm.invoke(q).content

for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i} (No RAG) ===\n{q}\n")
    print(no_rag(q))



=== Q1 (No RAG) ===
How can I tell if a text message is spam? Give 3 signs.

Here are three signs that can help you identify if a text message is spam:

1. **Unfamiliar Sender**: If the message comes from a number or contact you don't recognize, especially if it’s a random phone number or a short code, it could be spam. Legitimate businesses usually communicate from recognizable numbers or official channels.

2. **Urgent or Threatening Language**: Spam messages often use urgent language to create a sense of panic or fear, such as claiming your account will be suspended or that you’ve won a prize. Be cautious of messages that pressure you to act quickly or provide personal information.

3. **Requests for Personal Information**: If the message asks for sensitive information, such as passwords, Social Security numbers, or financial details, it’s likely a phishing attempt. Legitimate organizations typically do not request sensitive information via text message.

Always exercise caution an

### Baseline B: Simple RAG

In [51]:
def simple_rag(q: str, k: int = 5) -> str:
    matches = retrieve_sms(q, k=k)
    ctx_ids = [m["id"] for m in matches]
    ctx_text = "\n".join([f"- {rows[int(i)]['text']}" for i in ctx_ids])

    prompt = (
        "You are answering using retrieved SMS messages.\n"
        "Use the context to support your answer.\n\n"
        f"Context:\n{ctx_text}\n\n"
        f"Question:\n{q}\n"
    )
    return llm.invoke(prompt).content

for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i} (Simple RAG) ===\n{q}\n")
    print(simple_rag(q))



=== Q1 (Simple RAG) ===
How can I tell if a text message is spam? Give 3 signs.

You can identify a text message as spam by looking for the following signs:

1. **Unfamiliar Sender**: If the message comes from a number or sender you don't recognize, like the forwarded messages from "21870000" or messages with "Name Missing," it could be spam.

2. **Generic Content**: Spam messages often contain vague or generic content, such as the repeated phrase "some text missing" in the messages you received. This lack of personalization is a common characteristic of spam.

3. **Call to Action with Unusual Numbers**: Messages that prompt you to call back a specific number, especially if it seems unusual or is a premium rate number (like "09056242159"), can be a sign of spam. The mention of "cc100p/min" indicates potential charges, which is often a tactic used in spam messages.

=== Q2 (Simple RAG) ===
Write a short reply that refuses a spam offer politely.

Thank you for reaching out, but I'm not 

### Implement "RAG + metadata filter"
For spam-safety questions, retrieve only vectors with label="spam" so the context is more on-topic and less "random ham noise".

In [52]:
def retrieve_sms_spam_only(query: str, k: int = 5):
    q_vec = next(emb_model.embed([query])).tolist()
    res = index.query(
        vector=q_vec,
        top_k=k,
        include_metadata=True,
        filter={"label": "spam"},
    )
    return res["matches"]

def rag_spam_filter(q: str, k: int = 5) -> str:
    matches = retrieve_sms_spam_only(q, k=k)
    ctx_ids = [m["id"] for m in matches]
    ctx_text = "\n".join([f"- {rows[int(i)]['text']}" for i in ctx_ids])

    prompt = (
        "You are answering using retrieved SMS messages.\n"
        "The retrieved messages are spam-labeled examples.\n"
        "Use the context to support your answer.\n\n"
        f"Context:\n{ctx_text}\n\n"
        f"Question:\n{q}\n"
    )
    return llm.invoke(prompt).content

print(rag_spam_filter(questions[0]))


You can identify spam text messages by looking for the following signs:

1. **Unsolicited Offers or Promotions**: Messages that offer rewards, free services, or promotions that you did not sign up for are often spam. For example, the message stating "Claim ur 250 SMS messages-Text OK to 84025 now!" is a typical spam message promoting an unsolicited service.

2. **Urgency or Pressure to Act Quickly**: Spam messages often create a sense of urgency, prompting you to act quickly. The message that says, "Please call back on 09056242159 to retrieve your messages and matches" is designed to pressure you into calling a number without verifying its legitimacy.

3. **Unfamiliar Numbers or Senders**: Messages from unknown or suspicious numbers, especially those that appear to be automated or generic, are likely spam. For instance, the messages forwarded from "21870000" are from an unfamiliar sender and contain generic alerts about messages and matches, which is a common tactic used in spam texts.

### Implement "RAG + HyDE query rewriting"
use the LLM to write a "hypothetical" answer first, embed that, retrieve with that embedding, then answer with the retrieved context.

In [53]:
def hyde_rewrite(q: str) -> str:
    prompt = (
        "Write a short hypothetical answer to the question. "
        "Make it factual and include keywords that would appear in relevant SMS spam examples. "
        "Do not mention that this is hypothetical.\n\n"
        f"Question: {q}\n"
        "Hypothetical answer:"
    )
    return llm.invoke(prompt).content

def retrieve_sms_hyde(q: str, k: int = 5):
    hypo = hyde_rewrite(q)
    q_vec = next(emb_model.embed([hypo])).tolist()
    res = index.query(vector=q_vec, top_k=k, include_metadata=True)
    return hypo, res["matches"]

def rag_hyde(q: str, k: int = 5) -> str:
    hypo, matches = retrieve_sms_hyde(q, k=k)
    ctx_ids = [m["id"] for m in matches]
    ctx_text = "\n".join([f"- {rows[int(i)]['text']}" for i in ctx_ids])

    prompt = (
        "You are answering using retrieved SMS messages.\n"
        "Use the context to support your answer.\n\n"
        f"Question:\n{q}\n\n"
        f"HyDE draft:\n{hypo}\n\n"
        f"Context:\n{ctx_text}\n"
    )
    return llm.invoke(prompt).content

print(rag_hyde(questions[0]))


To identify if a text message is spam, look for these three signs:

1. **Unfamiliar Sender**: Messages from unknown numbers or random short codes are often spam. For example, the messages from "21870000" and "fullonsms.com" in the context provided are from unfamiliar sources, which raises a red flag.

2. **Urgent Language**: Spam texts frequently use urgent language to prompt immediate action. The message offering "250 SMS messages" includes phrases like "Claim ur 250 SMS messages" and urges you to "Text OK to 84025 now!" This creates a sense of urgency that is typical of spam.

3. **Suspicious Links**: Be wary of messages that contain links or instructions to text a specific number to claim something. The message from "Txt250.com" encourages users to join for a fee and includes terms and conditions, which can be a tactic to lure you into a subscription or phishing scheme.

Always verify the source before engaging with any unexpected messages.


### Eval

In [54]:
def run_eval():
    for i, q in enumerate(questions, 1):
        print("\n" + "="*80)
        print(f"Q{i}: {q}")

        print("\n--- A) No RAG ---")
        print(no_rag(q))

        print("\n--- B) Simple RAG ---")
        print(simple_rag(q))

        print("\n--- C) RAG + Spam Filter ---")
        print(rag_spam_filter(q))

        print("\n--- D) RAG + HyDE ---")
        print(rag_hyde(q))

run_eval()



Q1: How can I tell if a text message is spam? Give 3 signs.

--- A) No RAG ---
Here are three signs that can help you identify if a text message is spam:

1. **Unfamiliar Sender**: If the message comes from a number or contact you don't recognize, especially if it’s a random phone number or a short code, it could be spam. Legitimate businesses usually communicate from recognizable numbers or verified accounts.

2. **Urgent or Suspicious Language**: Spam messages often use urgent language to create a sense of panic or excitement, such as claiming you've won a prize, need to verify your account immediately, or must act quickly to avoid a negative consequence. Be cautious of messages that ask for personal information or prompt you to click on links.

3. **Poor Grammar and Spelling**: Many spam messages contain grammatical errors, misspellings, or awkward phrasing. If the message seems unprofessional or poorly written, it may be a sign that it’s not from a legitimate source.

Always exerc

## Comments

### Baseline: No RAG
The original LLM gives general safety advice. The answers are correct, but the content is very general. The answers do not use evidence from the SMS dataset (more like general advice, but not specific), so the advice is not grounded in the kinds of spam patterns that appear in the data.

### Simple RAG
Simple RAG improves the overall grounding. The model starts to mention real patterns from retrieved SMS, such as premium-rate call-back numbers, opt-out keywords, and generic alerts. This makes the answers more concrete and closer to real spam behavior. At the same time, retrieval can bring in some irrelevant messages, so some examples feel noisy.

### RAG + Advanced Change #1: Metadata filtering (spam-only retrieval)
This change improves relevance. For spam-related questions, the retriever only returns messages with `label="spam"`. The context becomes more consistent and the answers focus on real spam tactics like prize claims, "text OK to claim", and urgent call-back prompts. Compared with Simple RAG, the responses are less distracted by ham messages.

### RAG + Advanced Change #2: HyDE query rewriting
This change improves retrieval quality. HyDE first writes a short draft answer, then embeds that draft to retrieve better neighbors. In this task, HyDE tends to pull more "classic spam" examples that include keywords like prize, claim, free, call back, and short codes. The final answers stay similar in structure, but the supporting examples become more on-topic and easier to justify.

### Overall summary
RAG makes the answers more evidence-based because the model can point to real SMS examples. The spam-only filter reduces off-topic retrieval. HyDE makes the retrieved context more aligned with the user question.
